# Exercise 4 — format_completion_certificate

`format_completion_certificate` generates the final document that marks the end of the capstone: a Markdown certificate showing the project name, tagline, build phase checklist, completion percentage, tech stack, course sections applied, and a sign-off line. It is the last function you write on the last day.

In [ ]:
import datetime
from dataclasses import dataclass, field

@dataclass
class CapstoneSpec:
    name: str; tagline: str; domain: str; description: str
    sections_used: list; deliverables: list; tech_stack: list

@dataclass
class Phase:
    name: str; tasks: list; done: bool = False

@dataclass
class CapstoneReport:
    spec: CapstoneSpec
    phases: list = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.date.today().isoformat())
    completed_at: str = ""

_SPEC = CapstoneSpec(
    name          = "AI Trading Bot",
    tagline       = "Paper-trading bot with sentiment and technical signals.",
    domain        = "finance",
    description   = "End-to-end AI trading bot that fetches OHLCV data, computes "
                    "technical indicators, scores news sentiment with an LLM, applies "
                    "risk controls, and runs a daily paper-trading loop with logging.",
    sections_used = [
        "Section 3: Data & Analysis (pandas, SQLite)",
        "Section 4: Real Apps (FastAPI endpoint)",
        "Section 6: AI Agents (scheduling loop)",
        "Section 7: Finance & Trading (backtester, risk manager, paper trader)",
    ],
    deliverables  = [
        "paper_trader.py with buy/sell/portfolio_value",
        "bot_runner.py with daily scheduling and logging",
        "risk.py with stop-loss and drawdown controls",
        "Deployed FastAPI endpoint",
        "Portfolio case study",
    ],
    tech_stack    = ["Python", "pandas", "Ollama", "SQLite", "FastAPI"],
)
def validate_spec(spec):
    errors = []
    for field_name in ("name", "tagline", "domain", "description"):
        val = getattr(spec, field_name, "")
        if not isinstance(val, str) or not val.strip():
            errors.append(f"{field_name} must be a non-empty string")
    if len(spec.deliverables) < 3:
        errors.append(f"at least 3 deliverables required, got {len(spec.deliverables)}")
    if len(spec.tech_stack) < 2:
        errors.append(f"at least 2 tech stack items required, got {len(spec.tech_stack)}")
    if not spec.sections_used:
        errors.append("sections_used must reference at least one course section")
    return len(errors) == 0, errors
def generate_project_plan(spec):
    deliverable_list = "\n".join(f"- [ ] {d}" for d in spec.deliverables)
    tech_list        = "\n".join(f"- {t}"     for t in spec.tech_stack)
    sections_list    = "\n".join(f"- {s}"     for s in spec.sections_used)
    return (
        f"# {spec.name} — Implementation Plan\n\n"
        f"**{spec.tagline}**\n\n"
        f"## Overview\n\n{spec.description}\n\n"
        f"## Course Sections Applied\n\n{sections_list}\n\n"
        f"## Tech Stack\n\n{tech_list}\n\n"
        f"## Deliverables\n\n{deliverable_list}\n\n"
        f"## Build Phases\n\n"
        f"### Phase 1 — Plan\n- [ ] Finalise spec\n- [ ] Write gate tests\n\n"
        f"### Phase 2 — Build\n- [ ] Implement core AI\n- [ ] Wire pipeline\n\n"
        f"### Phase 3 — Test\n- [ ] Gate green\n- [ ] End-to-end test\n\n"
        f"### Phase 4 — Deploy\n- [ ] Write .env\n- [ ] Deploy\n\n"
        f"### Phase 5 — Document\n- [ ] README\n- [ ] Case study\n\n"
        f"### Phase 6 — Share\n- [ ] GitHub\n- [ ] Portfolio\n- [ ] Post\n"
    )
def default_phases(spec):
    ai_backend = next(
        (t for t in spec.tech_stack if t.lower() in ("ollama","llama","llamacpp")),
        "AI backend",
    )
    return [
        Phase("Plan",  [f"Finalise spec for {spec.name}", "Install packages", "Write gate tests"]),
        Phase("Build", ["Implement core AI", f"Wire {ai_backend}", "Build pipeline"]),
        Phase("Test",  ["Gate: all checks green", "Happy path", "Edge cases"]),
        Phase("Deploy",["Write .env", "Deploy to production", "Verify live URL"]),
        Phase("Document",["README", "Case study", "Demo video"]),
        Phase("Share", ["Push to GitHub", "Portfolio", "Post on LinkedIn"]),
    ]
def build_status(report):
    total = len(report.phases); done = sum(1 for p in report.phases if p.done)
    return {
        "phases": [{"name":p.name,"done":p.done,"n_tasks":len(p.tasks)} for p in report.phases],
        "total_phases": total, "completed_phases": done,
        "completion_pct": round(done / max(total,1) * 100.0, 1),
        "is_complete": (done == total and total > 0),
    }

def mark_phase_done(report, phase_name):
    for phase in report.phases:
        if phase.name == phase_name:
            phase.done = True; return True
    return False

def format_completion_certificate(report):
    """Generate a Markdown completion certificate.

    Structure:
      # 100 Days of AI — Capstone Completion
      ## {spec.name}
      **{spec.tagline}**
      {spec.description}
      ## Build Progress
      [x] Plan
      [ ] Build   ← done/not-done based on phase.done
      ...
      **Overall: {completion_pct}% complete**
      ## Tech Stack
      - item 1
      ## Course Sections Applied
      - section 1
      ---
      *Built during 100 Days of AI — The Complete AI Engineering Bootcamp*

    Returns:
        str — Markdown starting with "# 100 Days of AI"
    """
    spec   = report.spec
    status = build_status(report)
    phase_lines   = "\n".join(
        f"  {'[x]' if p['done'] else '[ ]'} {p['name']}"
        for p in status["phases"]
    )
    tech_lines    = "\n".join(f"- {t}" for t in spec.tech_stack)
    section_lines = "\n".join(f"- {s}" for s in spec.sections_used)
    # TODO: assemble the certificate string
    return ""


### Checks

In [ ]:
checks = 0

# Build a report with 3 phases done
report = CapstoneReport(spec=_SPEC, phases=default_phases(_SPEC))
for phase_name in ["Plan", "Build", "Test"]:
    mark_phase_done(report, phase_name)

# 1 — starts with # 100 Days of AI
try:
    cert = format_completion_certificate(report)
    assert cert.startswith("# 100 Days of AI"),         f"expected '# 100 Days of AI' start, got: {cert[:40]!r}"
    checks += 1; print("✅ 1 certificate starts with '# 100 Days of AI'")
except Exception as e:
    print("❌ 1:", e)

# 2 — project name and tagline present
try:
    cert = format_completion_certificate(report)
    assert _SPEC.name    in cert, f"name '{_SPEC.name}' not in certificate"
    assert _SPEC.tagline in cert, "tagline not in certificate"
    checks += 1; print(f"✅ 2 name '{_SPEC.name}' and tagline present")
except Exception as e:
    print("❌ 2:", e)

# 3 — done phases show [x]; not-done show [ ]
try:
    cert = format_completion_certificate(report)
    assert "[x] Plan"    in cert, "[x] Plan not in certificate"
    assert "[x] Build"   in cert, "[x] Build not in certificate"
    assert "[ ] Deploy"  in cert, "[ ] Deploy not in certificate"
    assert "[ ] Share"   in cert, "[ ] Share not in certificate"
    checks += 1; print("✅ 3 done phases show [x]; pending phases show [ ]")
except Exception as e:
    print("❌ 3:", e)

# 4 — completion percentage appears
try:
    cert = format_completion_certificate(report)
    status = build_status(report)
    pct_str = str(status["completion_pct"])
    assert pct_str in cert, f"completion_pct {pct_str!r} not in certificate"
    checks += 1; print(f"✅ 4 completion {pct_str}% appears in certificate")
except Exception as e:
    print("❌ 4:", e)

# 5 — sign-off line present; tech stack and sections appear
try:
    cert = format_completion_certificate(report)
    assert "100 Days of AI" in cert.split("---")[-1],         "sign-off line missing after ---"
    for t in _SPEC.tech_stack:
        assert t in cert, f"tech {t!r} not in certificate"
    checks += 1; print("✅ 5 sign-off line + all tech stack items present")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
